In [1]:
# !pip install moviepy
# !pip install accelerate

# !pip install packaging
# !pip install ninja
# !pip install flash-attn --no-build-isolation

import warnings
warnings.filterwarnings("ignore")

import os
import cv2
import torch
import shutil
from threading import Thread
from transformers import pipeline
from moviepy.editor import VideoFileClip
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

torch.random.manual_seed(0)

In [2]:
text = None
caption = None
LlmPipe = None

In [3]:
video_file = "/content/video.mp4"
audio_file = "/content/audio.mp3"
frames_folder = "/content/frames"

# Audio and Text Extraction

In [4]:
# Function to extract audio
def AudioAndTextCapture(video_path, audio_path):

  # Load the video clip
  video_clip = VideoFileClip(video_path)

  # Extract the audio from the video clip
  audio_clip = video_clip.audio

  # Write the audio to a separate file
  audio_clip.write_audiofile(audio_path)

  # Close the video and audio clips
  audio_clip.close()
  video_clip.close()

  pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    chunk_length_s=30,
    device=0
  )


  # Load the audio file
  with open(audio_path, "rb") as f:
    audio_data = f.read()

    # Perform ASR on the audio file
    prediction = pipe(audio_data)
    global text
    text = prediction['text']

# Frame (1 Frame Per Second) and Caption Extraction

In [5]:
# Function to extract frames
def FrameAndCaptionCapture(path, frames_folder):

    # Path to video file
    vidObj = cv2.VideoCapture(path)

    # Used as counter variable
    count = 0

    # checks whether frames were extracted
    success = True

    # Get the frame rate of the video
    fps = int(vidObj.get(cv2.CAP_PROP_FPS))

    # Create a directory to save frames if it doesn't exist
    if not os.path.exists(frames_folder):
        os.makedirs(frames_folder)

    while success:

        # vidObj object calls read function to extract frames
        success, image = vidObj.read()

        # Capture 1 frame per second
        if success and count % fps == 0:
            # Save the frame with frame-count
            cv2.imwrite(f"{frames_folder}/frame{count//fps}.jpg", image)
        count += 1

    # Load the image-to-text pipeline

    image_to_text = pipeline("image-to-text", model="nlpconnect/vit-gpt2-image-captioning", device=0)

    # Iterate through all files in the frames folder
    files = [os.path.join(frames_folder, filename) for filename in os.listdir(frames_folder)]

    # Generate caption for the image
    out = image_to_text(files)

    global caption
    caption = [i[0]['generated_text'] for i in out]

In [6]:
def LlmPipeline():
  global LlmPipe

  LlmModel = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-128k-instruct",
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="cuda",
  )

  LlmTokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct", device=0)

  LlmPipe = pipeline(
      "text-generation",
      model=LlmModel,
      tokenizer=LlmTokenizer,
  )

In [7]:
# Creating the threads
thread1 = Thread(target=FrameAndCaptionCapture, args=(video_file, frames_folder,))
thread2 = Thread(target=AudioAndTextCapture, args=(video_file, audio_file,))
thread3 = Thread(target=LlmPipeline)

# Starting the threads
thread1.start()
thread2.start()
thread3.start()

# Making threads joinable
thread1.join()
thread2.join()
thread3.join()

MoviePy - Writing audio in /content/audio.mp3


chunk:  26%|██▌       | 763/2929 [00:02<00:06, 315.98it/s, now=None]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

MoviePy - Done.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration. Please open a PR/issue to update `preprocessor_config.json` to use `image_processor_type` instead of `feature_extractor_type`. This warning will be removed in v4.40.
We strongly recommend passing in an `attention_mask` since your 

In [8]:
generation_args = {
    "max_new_tokens": 250,
    "return_full_text": False,
    "temperature": 0.0,
    "do_sample": False,
}

allcaptions='\n'.join(caption)

prompt= f''' use the following text that has been taken from video frames
     and audio's STT.
      The video frames have been captioned through an image captioning model,
      and the audio has been converted to text through whisper.
      The caption  text is: {allcaptions} and the stt text is {text}.
      So now write the summary precisely and accurately.'''

messages = [
    {"role": "user", "content": prompt}

]

output = LlmPipe(messages, **generation_args)

In [9]:
print(output[0]['generated_text'])

 The video frames depict a field scene with various animals, including dogs, cows, horses, zebras, and elephants. The dogs are described as working in a relay, taking turns to chase and potentially catch their prey. Wildebeests are portrayed as elusive targets, using tactics like confusing predators by running into groups of zebras and attempting to rejoin the main herd for safety. Despite the pack mentality of the dogs, one individual seems to defy this rule, attempting to rejoin its pack. The scene captures the dynamic interactions between predators and prey in a natural setting.


In [10]:
shutil.rmtree(frames_folder)
os.remove(audio_file)